<a href="https://colab.research.google.com/github/alexoviedo999/enterprise-document-rag/blob/main/RAG_System_Full_Code_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem Statement

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Business Context

As organizations grow and scale, they are often inundated with large volumes of data, reports, and documents that contain critical information for decision-making. In real-world business settings, such as venture capital firms like Andreesen Horowitz, business analysts are required to sift through large datasets, research papers, or reports to extract relevant information that impacts strategic decisions.

For instance, consider that you've just joined Andreesen Horowitz, a renowned venture capital firm, and you are tasked with analyzing a dense report like the Harvard Business Review's **"How Apple is Organized for Innovation."** Going through the report manually can be extremely time-consuming as the size and complexity of these report increases. However, by using **Semantic Search** and **Retrieval-Augmented Generation (RAG)** models, you can significantly streamline this process.

Imagine having the capability to directly ask questions like, “How does Apple structure its teams for innovation?” and get immediate, relevant answers drawn from the report. This ability to extract and organize specific insights quickly and accurately enables you to focus on higher-level analysis and decision-making, rather than being bogged down by information retrieval.

## Objective

The goal is to develop a RAG application that helps business analysts efficiently extract key insights from extensive reports, such as “How Apple is Organized for Innovation.”

Specifically, the system aims to:

- Answer user queries by retrieving relevant content directly from lengthy documents.

- Support natural-language interaction without requiring a full manual read-through.

- Act as an intelligent assistant that streamlines the report analysis process.

Through this solution, analysts can save time, improve productivity, and make faster, more informed strategic decisions

## Data Description

**How Apple is Organized for Innovation** - An article of 11 pages in pdf format

## Installing and Importing Necessary Libraries and Dependencies

Provide Python bindings for the llama.cpp library:

In [ ]:
!pip install llama-cpp-python --force-reinstall

  Using cached llama_cpp_python-0.3.22-py3-none-linux_x86_64.whl
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.7 kB)
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (22 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstall

 Installs several Python libraries required for this project:


In [ ]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain_openai==0.3.30 \
              faiss-cpu

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-chroma 1.1.0 requires chromadb<2.0.0,>=1.3.5, but you have chromadb 1.0.15 which is incompatible.
langchain-chroma 1.1.0 requires langchain-core<2.0.0,>=1.1.3, but you have langchain-core 0.3.84 which is incompatible.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.3.84 which is incompatible.
langgraph 1.1.9 requires langchain-core<2,>=1.3.0, but you have langchain-core 0.3.84 which is incompatible.


 Bring in necessary tools and functions from external libraries (above) so they can be used in below scripts:

In [ ]:
from langchain.chains import RetrievalQA
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.document_loaders import TextLoader
from huggingface_hub import hf_hub_download
from langchain.llms import LlamaCpp
# from langchain.llms import Ollama

Read PDF and extract to python variable:

In [ ]:
import fitz # PyMuPDF

# Define the path to the PDF document
pdf_path = '/content/HBR_How_Apple_Is_Organized_For_Innovation.pdf'

# Open the PDF document
document = fitz.open(pdf_path)

# Extract text from the PDF by iterating through pages
full_pdf_text = ""
for page_num in range(len(document)):
    page = document.load_page(page_num)
    full_pdf_text += page.get_text()

# Close the document
document.close()

Convert extracte PDF into specific chunk and overlap sizes:

In [ ]:
from langchain_core.documents import Document # Import Document class
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize a text splitter that uses the token encoder
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=500,
    chunk_overlap=50
)

# Create a LangChain Document object from the extracted text
# 'full_pdf_text' is assumed to be available from the previous cell (2388b0cb)
langchain_document = Document(page_content=full_pdf_text, metadata={"source": "HBR_How_Apple_Is_Organized_For_Innovation.pdf"})

# Split the LangChain Document object
text_chunks = text_splitter.split_documents([langchain_document])


In [ ]:
len(text_chunks)
715

715

Test chunk retrieval:

In [ ]:
#get the third chunk
text_chunks[3].page_content

'entirely new product categories such as the iPhone and the \nApple Watch, but also continually innovating within those \ncategories. Perhaps no product feature better reflects Apple’s \ncommitment to continuous innovation than the iPhone cam-\nera. When the iPhone was introduced, in 2007, Steve Jobs \ndevoted only six seconds to its camera in the annual keynote \nevent for unveiling new products. Since then iPhone camera \ntechnology has contributed to the photography industry \nwith a stream of innovations: High dynamic range imaging \n(2010), panorama photos (2012), True Tone flash (2013), opti-\ncal image stabilization (2015), the dual-lens camera (2016), \nportrait mode (2016), portrait lighting (2017), and night mode \n(2019) are but a few of the improvements.\nTo create such innovations, Apple relies on a structure \nthat centers on functional expertise. Its fundamental belief \nis that those with the most expertise and experience in a \ndomain should have decision rights for th

Instal and initialize embeding models:

In [ ]:

!pip install -q sentence-transformers
embeddings = SentenceTransformerEmbeddings(model_name="BAAI/bge-base-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Create vector DB using FAISS:

In [ ]:

vector_store = FAISS.from_documents(text_chunks, embedding=embeddings)

Confirm text chunk conversion into vector and stored in FAISS:

In [ ]:
vector_store.index.reconstruct(0)

array([-4.36183764e-03,  2.14977562e-02, -2.04677694e-02,  2.57130917e-02,
        8.43488201e-02, -3.52215655e-02,  2.77384240e-02,  2.18295772e-02,
       -3.14828306e-02, -3.56071442e-02, -4.03277241e-02, -7.68696517e-02,
       -3.99239734e-02,  1.96254756e-02, -3.66623886e-02,  3.19942646e-02,
        1.63672827e-02,  1.28385751e-02, -1.75524354e-02,  1.20822405e-02,
       -1.83997117e-02,  3.04091852e-02,  3.59396636e-02, -1.41272675e-02,
        4.73339781e-02, -4.59598862e-02,  3.45685259e-02, -1.52518256e-02,
       -1.32489428e-02, -4.46826452e-03,  2.66941134e-02, -5.41450754e-02,
        1.01213567e-02, -1.24545873e-03,  9.02715791e-03, -1.28184957e-03,
       -3.09436396e-03, -4.24807705e-02,  3.59259322e-02, -1.04435673e-02,
       -4.22881432e-02,  1.62809826e-02, -2.68659387e-02, -3.59725468e-02,
       -4.80926819e-02,  8.19278043e-03, -6.29612356e-02,  3.95891815e-02,
       -3.44665833e-02,  6.99856356e-02, -6.62973970e-02, -2.23842752e-03,
        2.39292756e-02, -

Define and download the model from HugginFace Hub:

In [ ]:
# Define the model repository and the file name from HuggingFace Hub
model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"
model_basename = "llama-2-13b-chat.Q5_K_M.gguf"

# Download the model file from HuggingFace Hub
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

Initialize the Llama-2 model using LlamaCpp to compress the model.

In [ ]:
# Initialize the LlamaCpp model with configuration
llm = LlamaCpp(
    model_path=model_path,
    temperature=0.01,
    top_p=0.95,
    verbose=False,
    n_ctx=4096,
    n_batch=2048,
    n_gpu_layers=43,
    max_tokens=512
)

Initiate orchestration of the RAG process:

In [ ]:
agent_colab = RetrievalQA.from_chain_type(
    llm=llm,
    verbose=False,  # Turned off to hide the long internal prompts
    chain_type="stuff",
    retriever=vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 6, "fetch_k": 20})
)

## Question Answering using LLM

### Question 1: Who are the authors of this article and who published this article?

In [ ]:
query="Who are the authors of this article and who published this article?"

In [ ]:
output_colab = agent_colab.run(query)
print("Answer:", output_colab)

Answer:  The authors of this article are Mr. Rosner, Mr. Hubel, and Ms. Haggerty. This article was published by Harvard Business Review in November–December 2020.


### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [ ]:
query="List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines."
output_colab = agent_colab.run(query)
print("Answer:", output_colab)

Answer: 

* Deep Expertise: Apple's leaders are expected to possess deep expertise in their areas and be able to meaningfully engage in all the work being done within their functions. This allows them to make informed decisions and provide guidance to their teams.

* Immersion in Details: Apple's leaders are expected to be immersed in the details of their functions, understanding the intricacies and nuances that can impact the success of a project or product. This allows them to identify potential issues and make informed decisions.

* Collaborative Debate: Apple's leaders are expected to engage in collaborative debate with other teams during collective decision-making. This allows for the exchange of ideas and perspectives, leading to more innovative solutions and better outcomes.


### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [ ]:
query="Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?"
output_colab = agent_colab.run(query)
print("Answer:", output_colab)

Answer:  Sure! The article highlights several examples of successful innovations at Apple that can be traced back to their leadership approach. One example is the development of the iPhone's portrait mode, which was driven by a fanatical attention to detail at the leadership level and intense collaborative debate among teams. Another example is the company's commitment to offering the best possible products, even if it means investing in areas that may not yield short-term profits. This approach has led to innovations such as the iPhone camera's ability to take portrait photos with bokeh, which was a feature previously only available on expensive single-lens reflex cameras. Additionally, Apple's functional organization structure allows experts to lead experts and make decisions based on their deep understanding of the technologies responsible for disruption, rather than relying on general managers who may not have the same level of expertise. This approach has allowed Apple to stay ahe

## Question Answering using the OpenAI LLM with Prompt Engineering

Set up OpenAI API client:

In [ ]:
from openai import OpenAI
import os
import json

file_name = '/content/config.json'
with open(file_name, 'r') as file:
    config = json.load(file)
    API_KEY = config.get("OPENAI_API_KEY")
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")

# Store API credentials in environment variables
if API_KEY:
    os.environ['OPENAI_API_KEY'] = API_KEY
else:
    print("Warning: OPENAI_API_KEY not found in config.json")

if OPENAI_API_BASE:
    os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE

# Initialize OpenAI client
client = OpenAI()
print("Client initialized.")

Client initialized.


Helper function that enables communication with the OpenAI chat model:

In [ ]:
# Define a function to get a response from the OpenAI chat model
def response(system_prompt, user_prompt, max_tokens=1000, temperature=0.2, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                    # Specify the model to use (GPT-4o in this case)
        messages=[
            {"role": "system", "content": system_prompt},                       # System prompt sets the assistant's behavior
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output (0 = deterministic)
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content

### Question 1: Who are the authors of this article and who published this article?

Using the OpenAI API

In [ ]:
query = "Who are the authors of this article and who published this article?"

# Retrieve more documents by increasing k to capture the author info
retrieved_docs = vector_store.similarity_search(query, k=8)

# Format the retrieved documents into a context string
context = "\n\n".join([d.page_content for d in retrieved_docs])

# Construct the RAG prompt for OpenAI
rag_user_prompt = f"Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n{context}\n\nQuestion: {query}\nHelpful Answer:"

# Call the response function with the RAG prompt
base_prompt_response_1 = response(system_prompt="You are a helpful assistant.", user_prompt=rag_user_prompt)
print("Answer:", base_prompt_response_1)

Answer: The authors of the article are Joel M. Podolny and Morten T. Hansen. The article was published by Harvard Business Review in November–December 2020.


### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [ ]:
query = "List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines."

# Retrieve more documents by increasing k
retrieved_docs = vector_store.similarity_search(query, k=8)

# Format the retrieved documents into a context string
context = "\n\n".join([d.page_content for d in retrieved_docs])

# Construct the RAG prompt for OpenAI
rag_user_prompt = f"Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n{context}\n\nQuestion: {query}\nHelpful Answer:"

# Call the response function with the RAG prompt
base_prompt_response_1 = response(system_prompt="You are a helpful assistant.", user_prompt=rag_user_prompt)
print("Answer:", base_prompt_response_1)

Answer: - **Deep Expertise**: Leaders at Apple are expected to have specialized knowledge in their respective fields, allowing them to engage meaningfully with the work being done and make informed decisions.

- **Immersion in Details**: Apple leaders are required to be deeply involved in the specifics of their functions, ensuring that they understand the intricacies and nuances that can impact product quality and innovation.

- **Willingness to Collaboratively Debate**: Leaders must be open to discussing and debating ideas with peers from different functions, fostering a culture of collaboration that leads to well-rounded decision-making and innovation.


### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [ ]:
query = "Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?"

# Retrieve more documents by increasing k
retrieved_docs = vector_store.similarity_search(query, k=8)

# Format the retrieved documents into a context string
context = "\n\n".join([d.page_content for d in retrieved_docs])

# Construct the RAG prompt for OpenAI
rag_user_prompt = f"Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n{context}\n\nQuestion: {query}\nHelpful Answer:"

# Call the response function with the RAG prompt
base_prompt_response_1 = response(system_prompt="You are a helpful assistant.", user_prompt=rag_user_prompt)
print("Answer:", base_prompt_response_1)

Answer: One specific example from the article highlighting Apple's approach to leadership leading to successful innovations is the development of the dual-lens camera with portrait mode in the iPhone 7 Plus. This innovation required collaboration among various specialist teams and was a significant risk for Apple. The decision to introduce this feature was driven by the expertise of Paul Hubel, a senior leader who played a central role in the project. His team took a considerable gamble, betting that users would value the enhanced camera capabilities enough to justify the higher cost of the phone. The success of the dual-lens camera not only became a defining feature of the iPhone 7 Plus but also enhanced the credibility and reputation of Hubel and his team within the company.

Another example is Apple's meticulous attention to product design, particularly in the shape of product corners. The use of a "squircle" instead of a standard rounded rectangle demonstrates Apple's commitment to

## Data Preparation for RAG

### Loading the Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

# Set the path to the PDF file
manual_pdf_path = "/content/HBR_How_Apple_Is_Organized_For_Innovation.pdf"                       # Path to the medical diagnosis manual PDF

# Load the PDF using LangChain's PyPDFLoader
pdf_loader = PyMuPDFLoader(manual_pdf_path)                                     # Initialize the PDF loader with the file path

# Extract content from the PDF
applePdf = pdf_loader.load()

Confirm loaded PDF

In [ ]:
print(f"Successfully loaded {len(applePdf)} pages from the PDF.")

Successfully loaded 11 pages from the PDF.


### Data Chunking

Chunk the PDF into Manageable Text Sections Using a Token-Based Splitter

In [ ]:
# Initialize a text splitter that uses OpenAI's token encoder
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',                                                # Encoding used by popular LLMs
    chunk_size=800,                                                             # Each chunk will have up to 800 tokens
    chunk_overlap=100                                                           # 100 tokens will overlap between consecutive chunks (for context continuity)
)

Split the Loaded PDF into Chunks for Further Processing

In [ ]:
# Use the text splitter to divide the PDF content into smaller chunks
document_chunks = pdf_loader.load_and_split(text_splitter)

Check the Number of Chunks Created

In [ ]:
len(document_chunks)

17

### Embedding

 Initialize OpenAI embeddings model and use it to generate vector representations for text:

In [ ]:
from langchain_openai import OpenAIEmbeddings

# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=API_KEY,                                                     # Your OpenAI API key for authentication
    openai_api_base=OPENAI_API_BASE                                             # The OpenAI API base URL endpoint
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))

Dimension of the embedding vector  1536


### Vector Database


Setup Vector Store Directory

In [ ]:
# Creating a folder for saving the vector DB so it persists between runs
out_dir = 'apple_db'                                                          # Directory to store the persistent vector database

# Create the directory if it doesn't exist
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

Create Vector Store from Documents

In [ ]:
!pip install langchain_chroma
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    document_chunks,                                                            # Documents to index
    embedding_model,                                                            # Embedding model for converting text to vectors
    persist_directory=out_dir                                                   # Save vector DB files here
)

  Using cached chromadb-1.5.8-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
  Using cached langchain_core-1.3.2-py3-none-any.whl.metadata (4.4 kB)
Using cached chromadb-1.5.8-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (23.2 MB)
Using cached langchain_core-1.3.2-py3-none-any.whl (542 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.84
    Uninstalling langchain-core-0.3.84:
      Successfully uninstalled langchain-core-0.3.84
  Attempting uninstall: chromadb
    Found existing installation: chromadb 1.0.15
    Uninstalling chromadb-1.0.15:
      Successfully uninstalled chromadb-1.0.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.3.27 requires langchain-core<1.0.0,>=0.3.66, but you have langchain-core 1.3.2 which is incompatible.
langchain 0.3.27 req

Load Vector Store

In [ ]:
vectorstore = Chroma(
    persist_directory=out_dir,                                                  # Load existing vector DB files
    embedding_function=embedding_model                                          # Use the same embedding function for queries
)

Explore Vector Store and Perform Searches

In [ ]:
# Inspect the embedding function in use
vectorstore.embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7bded48c6900>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7bdf207ced80>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base='https://aibe.mygreatlearning.com/openai/v1', openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

### Retriever

Convert Vector Store into a Retriever and Retrieve Relevant Documents

In [ ]:
# Wrap the vector store into a retriever object
# We will use MMR (Maximal Marginal Relevance) to fetch diverse but relevant chunks
retriever = vectorstore.as_retriever(
    search_type='mmr',                                                   # MMR ensures diversity in the retrieved documents
    search_kwargs={'k': 6, 'fetch_k': 20}                                # Fetch 20 chunks, then pick the top 6 most diverse & relevant
)

System and User Prompt Template

### Response Function

In [ ]:
qna_system_message = """
You are a helpful and accurate AI assistant for business analysts. Your task is to answer questions based solely on the provided context from the 'How Apple is Organized for Innovation' report. Do not use any outside knowledge. If the answer is not present in the provided context, state that you don't know."

This prompt guides the model to act as a factual assistant, relying only on the document for answers, which is crucial for a RAG system.
"""

In [ ]:
# Define the user message template
qna_user_message_template = """
###Context
Here are some excerpts from the 'How Apple is Organized for Innovation' report and their sources that are relevant to the question mentioned below:
{context}

###Question
{question}
"""

Response Function

In [ ]:
def generate_rag_response(user_input, max_tokens=1000, temperature=0.75, top_p=0.95):
    global qna_system_message, qna_user_message_template

    # Retrieve relevant document chunks using the modern 'invoke' method
    relevant_document_chunks = retriever.invoke(user_input)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = "\n\n".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": qna_system_message},
                {"role": "user", "content": user_message}
            ],
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Question 1: Who are the authors of this article and who published this article?

In [ ]:
question_1 = "Who are the authors of this article and who published this article?"
response_with_rag_1 = generate_rag_response(question_1)
response_with_rag_1

'The authors of the article are Joel M. Podolny and Morten T. Hansen. The article was published by Harvard Business Review.'

### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [ ]:
question_2 = "List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines."
response_with_rag_2 = generate_rag_response(question_2)
response_with_rag_2

'- **Deep Expertise**: Leaders at Apple are expected to have significant knowledge and skills in their specific functional areas, allowing them to effectively engage with the work being done and make informed decisions.\n\n- **Immersion in Details**: Apple managers are involved in the specifics of their functions, ensuring they understand the intricacies that affect their teams and projects, which enhances decision-making quality.\n\n- **Willingness to Collaboratively Debate**: Leaders are encouraged to engage in discussions across functions, advocating for their views while being open to changing their minds based on evidence, fostering a culture of collective decision-making.'

### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [ ]:
question_3 = "Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?"
response_with_rag_3 = generate_rag_response(question_3)
response_with_rag_3

'One specific example from the article where Apple\'s approach to leadership has led to successful innovations is the development of the dual-lens camera with portrait mode for the iPhone 7 Plus. This innovation required collaboration among dozens of specialist teams, including the video engineering team, which was responsible for the low-level software that controls sensor and camera operations. The collaborative debate involved various functions where team members could disagree, push back, and build on each other\'s ideas to arrive at the best solution. Ultimately, this collaboration resulted in a feature that was central to Apple\'s marketing of the iPhone 7 Plus and proved to be a major reason for users choosing to buy the phone.\n\nAnother example is the leadership of Roger Rosner, who has risen through the ranks at Apple and exemplifies the concept of "experts leading experts." His deep expertise in software applications enabled him to effectively guide a rapidly growing team an

In [ ]:
import json

notebook_filename = 'Alejandro_Oviedo_Learners_Notebook_Full_Code.ipynb'
cleaned_filename = 'Cleaned_Notebook.ipynb'

try:
    # Load the notebook
    with open(notebook_filename, 'r', encoding='utf-8') as f:
        nb = json.load(f)

    # Remove the widgets metadata at the notebook level
    if 'metadata' in nb and 'widgets' in nb['metadata']:
        del nb['metadata']['widgets']

    # Clean outputs
    for cell in nb.get('cells', []):
        if 'outputs' in cell:
            for output in cell['outputs']:
                # Remove widget state
                if 'data' in output and 'application/vnd.jupyter.widget-view+json' in output['data']:
                    del output['data']['application/vnd.jupyter.widget-view+json']
                # Remove verbose LangChain logs from text/stdout
                if 'text' in output and isinstance(output['text'], list):
                    cleaned_text = []
                    for line in output['text']:
                        if not any(phrase in line for phrase in ['> Entering new', 'Prompt after formatting:', 'Use the following pieces of context']):
                            cleaned_text.append(line)
                    output['text'] = cleaned_text

    # Save the cleaned notebook
    with open(cleaned_filename, 'w', encoding='utf-8') as f:
        json.dump(nb, f, indent=1)

    print(f"Successfully cleaned widget metadata and verbose logs, and saved to {cleaned_filename}")
except FileNotFoundError:
    print(f"Error: The file {notebook_filename} was not found. Please ensure it is saved in the current directory.")

Successfully cleaned widget metadata and verbose logs, and saved to Cleaned_Notebook.ipynb


In [ ]:
!jupyter nbconvert --to html /content/Cleaned_Notebook.ipynb --output /content/Cleaned_Notebook-1.html

[NbConvertApp] Converting notebook /content/Cleaned_Notebook.ipynb to html
/usr/local/share/jupyter/nbconvert/templates/base/display_priority.j2:32: UserWarning: Your element with mimetype(s) dict_keys(['application/vnd.colab-display-data+json']) is not able to be represented.
  {%- elif type == 'text/vnd.mermaid' -%}
[NbConvertApp] Writing 440211 bytes to /content/Cleaned_Notebook-1.html


## Actionable Insights and Business Recommendations

- The open ai model is much more turn-key
- Adjusting the chunk size does matter. Adjusting k is helpful, setting it a 4 was a good mid-range that also saves on tokens
- Limiting the LLM to only respond if it has the specific and correct information is essential, the results are in the question 3, the response is such that the model responded "I don't know" and specifically said why.
- Model Flexibility: By implementing both a local, open-source model (Llama-2 via llama.cpp) and a commercial cloud API (OpenAI GPT-4o-mini), you have created a flexible architecture. You can use the local model for highly sensitive internal documents to guarantee data privacy, and switch to OpenAI when you need maximum reasoning capabilities and speed for non-sensitive data.
- Upgrading to a token-based splitter (cl100k_base), adjusting chunk sizes, and implementing Maximal Marginal Relevance (MMR) search ensures that the models receive a diverse and highly relevant set of text chunks, drastically reducing hallucinations.
- System prompts act as guardrails. By explicitly instructing the model to rely only on the provided context and to admit when it doesn't know the answer, you tailor the system for business reliability.
- As the document library grows from one 11-page PDF to hundreds of reports, there may be a need to upgrade from a local Chroma/FAISS database to a dedicated, enterprise vector database like Pinecone.


<font size=6 color='#4682B4'>Power Ahead</font>
___